# Breast Tumor Classification with ResNet50

This notebook trains a deep learning model to classify breast ultrasound (BUS) images as malignant or benign. The dataset provides `train.xlsx` and `test.xlsx`, so the model is trained only on the training split and evaluated only on the testing split.


## 1. Import Libraries and Set Paths

This cell imports the libraries used throughout the notebook and defines the dataset paths. The model uses the resized `224x224` image folder that was created earlier.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Dataset paths
DATASET_DIR = Path('/data/ramialle/datasets2/Dataset_BUSI_with_GT_Clean')
IMAGE_DIR = DATASET_DIR / 'images_224'
TRAIN_XLSX = DATASET_DIR / 'train.xlsx'
TEST_XLSX = DATASET_DIR / 'test.xlsx'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20

# In the Excel files: 0 = malignant and 1 = benign.
CLASS_NAMES = {0: 'malignant', 1: 'benign'}
print(f'Using images from: {IMAGE_DIR}')


## 2. Read the Excel Train and Test Splits

This cell reads `train.xlsx` and `test.xlsx`. A small Excel reader is included so the notebook does not depend on `pandas` or `openpyxl` just to load the two split files.


In [ ]:
def _column_index(cell_reference):
    """Convert an Excel cell reference like A1 or B12 into a zero-based column index."""
    letters = ''.join(ch for ch in cell_reference if ch.isalpha())
    index = 0
    for char in letters:
        index = index * 26 + (ord(char.upper()) - ord('A') + 1)
    return index - 1


def read_xlsx_rows(path):
    """Read the first worksheet of a simple .xlsx file into a list of dictionaries."""
    namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

    with ZipFile(path) as workbook_zip:
        shared_strings = []
        if 'xl/sharedStrings.xml' in workbook_zip.namelist():
            root = ET.fromstring(workbook_zip.read('xl/sharedStrings.xml'))
            for item in root.findall('main:si', namespace):
                text_parts = [node.text or '' for node in item.findall('.//main:t', namespace)]
                shared_strings.append(''.join(text_parts))

        sheet = ET.fromstring(workbook_zip.read('xl/worksheets/sheet1.xml'))
        raw_rows = []
        for row in sheet.findall('.//main:sheetData/main:row', namespace):
            values = []
            for cell in row.findall('main:c', namespace):
                column = _column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column:
                    values.append('')

                cell_type = cell.attrib.get('t')
                value_node = cell.find('main:v', namespace)
                inline_node = cell.find('main:is/main:t', namespace)

                if cell_type == 's' and value_node is not None:
                    value = shared_strings[int(value_node.text)]
                elif cell_type == 'inlineStr' and inline_node is not None:
                    value = inline_node.text or ''
                elif value_node is not None:
                    value = value_node.text or ''
                else:
                    value = ''

                values[column] = value
            raw_rows.append(values)

    header = [str(value).strip() for value in raw_rows[0]]
    rows = []
    for row in raw_rows[1:]:
        row = row + [''] * (len(header) - len(row))
        record = dict(zip(header, row))
        if record.get('Image'):
            record['Image'] = str(record['Image']).strip()
            record['Label'] = int(float(record['Label']))
            record['path'] = IMAGE_DIR / record['Image']
            record['class_name'] = CLASS_NAMES[record['Label']]
            rows.append(record)
    return rows


train_rows = read_xlsx_rows(TRAIN_XLSX)
test_rows = read_xlsx_rows(TEST_XLSX)

print(f'Training samples: {len(train_rows)}')
print(f'Testing samples: {len(test_rows)}')
print('Example training row:', train_rows[0])
print('Example testing row:', test_rows[0])


## 3. Verify Images and Class Counts

This cell checks that all images referenced by the Excel files exist and are exactly `224x224`. It also prints the benign/malignant counts in each split.


In [ ]:
def summarize_split(rows, split_name):
    labels = [row['Label'] for row in rows]
    counts = {CLASS_NAMES[label]: labels.count(label) for label in sorted(CLASS_NAMES)}
    print(f'{split_name} class counts:', counts)


def verify_images(rows, split_name):
    missing = [row['path'] for row in rows if not row['path'].exists()]
    if missing:
        raise FileNotFoundError(f'{split_name} has missing image files. First missing file: {missing[0]}')

    bad_sizes = []
    for row in rows:
        with Image.open(row['path']) as img:
            if img.size != IMG_SIZE:
                bad_sizes.append((row['Image'], img.size))

    if bad_sizes:
        raise ValueError(f'{split_name} has non-224x224 images. Examples: {bad_sizes[:5]}')

    print(f'{split_name}: all images exist and are {IMG_SIZE[0]}x{IMG_SIZE[1]}.')


summarize_split(train_rows, 'Train')
summarize_split(test_rows, 'Test')
verify_images(train_rows, 'Train')
verify_images(test_rows, 'Test')


## 4. Build TensorFlow Datasets

This cell converts the Excel rows into efficient TensorFlow datasets. Images are decoded, resized to `224x224` as a safety check, and preprocessed using the official ResNet50 preprocessing function.


In [ ]:
def load_and_preprocess_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_png(image_bytes, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)
    return image, tf.cast(label, tf.float32)


def make_dataset(rows, training=False):
    paths = [str(row['path']) for row in rows]
    labels = [row['Label'] for row in rows]

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(buffer_size=len(rows), seed=SEED, reshuffle_each_iteration=True)

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset


train_ds = make_dataset(train_rows, training=True)
test_ds = make_dataset(test_rows, training=False)

for images, labels in train_ds.take(1):
    print('Image batch shape:', images.shape)
    print('Label batch shape:', labels.shape)


## 5. Create the ResNet50 Model

This cell loads ResNet50 with pretrained ImageNet weights and replaces the original classification head with a binary classifier for benign vs malignant BUS images.


In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
)

# Freeze the pretrained feature extractor for the first training stage.
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.35)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.25)(x)

# Sigmoid output: probability that the image is benign, because benign is label 1.
output = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()


## 6. Train the Model

This cell trains the model on the training split and evaluates testing loss after every epoch using the testing split as validation data. Class weights help balance the benign and malignant classes.


In [ ]:
train_labels = np.array([row['Label'] for row in train_rows])
class_counts = np.bincount(train_labels, minlength=2)
class_weight = {
    class_id: len(train_labels) / (len(CLASS_NAMES) * count)
    for class_id, count in enumerate(class_counts)
}
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in class_weight.items()})

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7),
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 7. Plot Training and Testing Loss

This cell plots the training loss and testing loss for each epoch. If both losses stay high, the model may be underfitting. If training loss keeps decreasing while testing loss rises, the model may be overfitting.


In [ ]:
epochs_ran = range(1, len(history.history['loss']) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history.history['loss'], marker='o', label='Training loss')
plt.plot(epochs_ran, history.history['val_loss'], marker='o', label='Testing loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history.history['accuracy'], marker='o', label='Training accuracy')
plt.plot(epochs_ran, history.history['val_accuracy'], marker='o', label='Testing accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Evaluate with Accuracy, Recall, Precision, F1 Score, and Confusion Matrix

This cell evaluates the trained model on the testing set. Metrics are computed for both classes so that malignant and benign performance can be inspected separately.


In [ ]:
y_true = np.array([row['Label'] for row in test_rows])
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = np.mean(y_pred == y_true)
confusion = np.zeros((2, 2), dtype=int)
for true_label, pred_label in zip(y_true, y_pred):
    confusion[true_label, pred_label] += 1

print(f'Test accuracy: {accuracy:.4f}')
print('\nConfusion matrix rows=true, columns=predicted')
print('Labels:', [CLASS_NAMES[0], CLASS_NAMES[1]])
print(confusion)

print('\nPer-class metrics:')
for class_id, class_name in CLASS_NAMES.items():
    tp = confusion[class_id, class_id]
    fp = confusion[:, class_id].sum() - tp
    fn = confusion[class_id, :].sum() - tp

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    print(f'{class_name:10s} precision={precision:.4f} recall={recall:.4f} f1={f1:.4f}')

macro_f1_values = []
for class_id in CLASS_NAMES:
    tp = confusion[class_id, class_id]
    fp = confusion[:, class_id].sum() - tp
    fn = confusion[class_id, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    macro_f1_values.append(2 * precision * recall / (precision + recall) if (precision + recall) else 0.0)

print(f'\nMacro F1 score: {np.mean(macro_f1_values):.4f}')


## 9. Display Each Testing Image with True and Predicted Class

This final cell loops through the testing images and displays the image filename, true label, predicted class, and predicted benign probability. Misclassified images are highlighted in red titles.


In [ ]:
for row, probability, prediction in zip(test_rows, y_prob, y_pred):
    true_name = CLASS_NAMES[row['Label']]
    predicted_name = CLASS_NAMES[int(prediction)]
    title_color = 'green' if prediction == row['Label'] else 'red'

    image = Image.open(row['path']).convert('RGB')
    plt.figure(figsize=(4, 4))
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    plt.title(
        f"{row['Image']}\nTrue: {true_name} | Predicted: {predicted_name}\nBenign probability: {probability:.3f}",
        color=title_color,
        fontsize=10,
    )
    plt.show()
